In [27]:
from pathlib import Path
import math
import pandas as pd
import re
from datetime import datetime

# =========================
# 기본 경로 설정
# =========================
# FCT testlog 상위 경로 (필요하면 여기만 수정)
BASE_DIR = Path(r"C:\Users\user\Desktop\machinlog\FCT")

# 재실행 시 중복 방지를 위한 이력 CSV 파일 경로
HISTORY_CSV_PATH = Path(r"C:\Users\user\Desktop\c1_fct_testlog_detail_jason_processing.csv")

# 샘플 결과(상위 1000행) 엑셀 저장 경로
SAMPLE_EXCEL_PATH = Path(r"C:\Users\user\Desktop\c1_fct_testlog_detail_result_sample1000.xlsx")


# =========================
# 유틸 함수들
# =========================
def read_lines_with_encodings(file_path: Path):
    """
    여러 인코딩(cp949, utf-8, utf-8-sig)을 시도해서
    글자가 깨지지 않도록 안전하게 읽기.
    """
    for enc in ("cp949", "utf-8", "utf-8-sig"):
        try:
            with file_path.open("r", encoding=enc) as f:
                return f.readlines()
        except UnicodeDecodeError:
            continue

    # 그래도 안 되면 마지막에만 ignore 사용
    with file_path.open("r", encoding="utf-8", errors="ignore") as f:
        return f.readlines()


def extract_yyyymmdd_from_name(name: str) -> str:
    """
    파일명에서 YYYYMMDD 추출.

    1) 파일명 첫 번째 '-' 위치 찾기
    2) 파일명 두 번째 '_' 위치 찾기
    3) 그 사이 문자열에서 마지막 8자리 숫자(YYYYMMDD)를 찾기
    4) 없으면 파일명 전체에서 20으로 시작하는 8자리 숫자 찾기
    5) 그래도 없으면 "" 리턴
    """
    dash_pos = name.find("-")
    # 첫 번째 '_' 와 두 번째 '_' 찾기
    first_us_pos = name.find("_")
    second_us_pos = -1
    if first_us_pos != -1:
        second_us_pos = name.find("_", first_us_pos + 1)

    candidate = ""
    if dash_pos != -1 and second_us_pos != -1 and second_us_pos > dash_pos:
        # 첫 번째 '-' 이후 ~ 두 번째 '_' 사이 문자열
        between = name[dash_pos + 1:second_us_pos]
        # 그 안에서 마지막 8자리 숫자 찾기
        m = re.findall(r"(\d{8})", between)
        if m:
            candidate = m[-1]

    # 위에서 못 찾았으면 파일명 전체에서 20으로 시작하는 8자리 숫자 검색
    if not candidate:
        m2 = re.findall(r"(20\d{6})", name)
        if m2:
            candidate = m2[-1]

    return candidate


def parse_filename(filepath: Path):
    """
    파일명에서 Barcode information, YYYYMMDD 추출

    - Barcode information : 확장자 제거한 후, 첫 번째 '_' 앞까지
    - YYYYMMDD : extract_yyyymmdd_from_name() 규칙 사용
    """
    name = filepath.stem  # 확장자 제거

    # 1) Barcode information
    if "_" in name:
        barcode = name.split("_", 1)[0]
    else:
        barcode = name  # '_'가 없으면 전체를 바코드로

    # 2) YYYYMMDD
    yyyymmdd = extract_yyyymmdd_from_name(name)

    return barcode, yyyymmdd


def parse_time_line(line: str):
    """
    로그 한 줄에서 [hh:mm:ss.ss] 와 Test_item 추출
    - [hh:mm:ss.ss] 가 없으면 None 반환
    - Test_item 내부의 2개 이상 공백은 1개로 축소
    """
    m = re.search(r"\[(\d{2}:\d{2}:\d{2}\.\d{2})\]\s+(.*)", line)
    if not m:
        return None, None

    time_str = m.group(1)  # "hh:mm:ss.ss"
    test_item_raw = m.group(2)

    # 내용안에 공백 1개까지 허용, 2개 이상의 공백은 1개로 축소
    test_item = re.sub(r"\s{2,}", " ", test_item_raw).strip()

    return time_str, test_item


def time_to_seconds(time_str: str) -> float:
    """
    "hh:mm:ss.ss" → 초(float)로 변환
    """
    t = datetime.strptime(time_str, "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1_000_000


def get_end_time_str(last_time_str: str) -> str:
    """
    마지막 [hh:mm:ss.ss] 에서 hh:mm:ss 만 추출해 문자열로 반환
    """
    t = datetime.strptime(last_time_str, "%H:%M:%S.%f")
    return t.strftime("%H:%M:%S")


def process_one_file(file_path: Path):
    """
    하나의 txt 파일을 파싱해서 row 리스트 반환
    각 row는 아래 컬럼을 가짐:
    - YYYYMMDD
    - End time
    - Barcode information
    - Test_item
    - Test_Time      : 각 이벤트의 [hh:mm:ss.ss]
    - Test_item_CT   : 이전 이벤트와의 시간 차(초)
    """
    barcode, yyyymmdd = parse_filename(file_path)

    # 파일 내용 읽기 (인코딩 자동 처리)
    lines = read_lines_with_encodings(file_path)

    events = []  # (time_str, test_item)

    for line in lines:
        time_str, test_item = parse_time_line(line)
        if time_str is None:
            # [hh:mm:ss.ss] 가 없는 행은 완전히 무시
            continue
        events.append((time_str, test_item))

    # 유효한 타임스탬프가 하나도 없으면 이 파일은 스킵
    if not events:
        return []

    # End time: 마지막 이벤트의 시간에서 hh:mm:ss 추출
    last_time_str = events[-1][0]
    end_time = get_end_time_str(last_time_str)

    # Test_item_CT 계산, Test_Time 설정
    rows = []
    prev_sec = None

    for time_str, test_item in events:
        cur_sec = time_to_seconds(time_str)

        if prev_sec is None:
            ct_value = None  # 첫 번째 Test_item은 NULL
        else:
            diff = cur_sec - prev_sec
            # 만약 시간 차가 음수면(자정 넘어간 경우 등) 24시간 더해줌
            if diff < 0:
                diff += 24 * 3600
            ct_value = round(diff, 2)

        prev_sec = cur_sec

        row = {
            "YYYYMMDD": yyyymmdd,
            "End time": end_time,
            "Barcode information": barcode,
            "Test_item": test_item,
            "Test_Time": time_str,      # 예시대로 [hh:mm:ss.ss] 그대로
            "Test_item_CT": ct_value,   # 이전 이벤트와의 시간 차(초)
        }
        rows.append(row)

    return rows


def is_valid_deep_fct_path(p: Path) -> bool:
    """
    BASE_DIR 기준 상대경로가
    YYYY/MM/DD/어떤폴더/파일 구조인지 확인.

    예) OK:
      2025/10/01/FORD A+C PD NONE 35654264/BA1WJ....txt
      → parts = ['2025','10','01','FORD A+C PD NONE 35654264','BA1WJ....txt']

    예) NG:
      2025/10/20251001_FCT1_Machine_Log.txt
      → parts = ['2025','10','20251001_FCT1_Machine_Log.txt']  (길이 3 → 제외)
    """
    try:
        rel = p.relative_to(BASE_DIR)
    except ValueError:
        return False

    parts = rel.parts  # ('2025', '10', '01', '...', 'file.txt') 등

    # 최소 구조: YYYY / MM / DD / (폴더) / 파일 → 4개 이상
    if len(parts) < 4:
        return False

    year, month, day = parts[0], parts[1], parts[2]

    if not (year.isdigit() and len(year) == 4):
        return False
    if not (month.isdigit() and len(month) == 2):
        return False
    if not (day.isdigit() and len(day) == 2):
        return False

    return True


# =========================
# 메인 로직
# =========================
def main():
    # 1) 전체 TXT 파일 스캔 (우선 전부 찾고, 나중에 YYYY/MM/DD/폴더/파일 구조만 필터링)
    all_found_txt_files = list(BASE_DIR.rglob("*.txt"))

    # YYYY/MM/DD/무슨폴더/파일 구조만 남기기
    all_txt_files = [p for p in all_found_txt_files if is_valid_deep_fct_path(p)]

    print(f"1) TXT 파일 스캔 완료 → 총 파일 수집: {len(all_txt_files)}개")

    # 2) 이력 CSV 로드 (이미 처리한 파일 제외)
    if HISTORY_CSV_PATH.exists():
        history_df = pd.read_csv(HISTORY_CSV_PATH)
        processed_file_paths = set(history_df["file_path"].astype(str))
        print(f"2) 이전에 처리한 파일 수: {len(processed_file_paths)}개")
    else:
        history_df = None
        processed_file_paths = set()
        print("2) 이력 CSV 없음 → 처음 실행으로 간주")

    target_files = [p for p in all_txt_files if str(p) not in processed_file_paths]
    total = len(target_files)
    print(f"3) 이번에 새로 처리할 대상 파일 수: {total}개")

    all_rows = []
    new_history_rows = []

    if total > 0:
        print("4) TXT 파일 처리 시작...")

    for i, file_path in enumerate(target_files, start=1):
        try:
            rows = process_one_file(file_path)
        except Exception as e:
            print(f"   [ERROR] 파일 처리 중 에러 발생: {e}")
            continue

        if not rows:
            # 유효 로그 없으면 이력에도 남기지 않음
            continue

        all_rows.extend(rows)
        new_history_rows.append({
            "file_path": str(file_path),
            "processed_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

        # 1000개 단위로만 진행상황 출력 (파일 경로 X)
        if (i % 1000 == 0) or (i == total):
            print(f"   → 현재 {i}/{total} 파일 처리 완료")

    if total > 0:
        print("4) TXT 파일 처리 완료")

    # DataFrame 생성
    if all_rows:
        df = pd.DataFrame(
            all_rows,
            columns=[
                "YYYYMMDD",
                "End time",
                "Barcode information",
                "Test_item",
                "Test_Time",
                "Test_item_CT",
            ],
        )
        print(f"\n5) 총 생성된 row 수: {len(df)}개")
    else:
        df = pd.DataFrame(
            columns=[
                "YYYYMMDD",
                "End time",
                "Barcode information",
                "Test_item",
                "Test_Time",
                "Test_item_CT",
            ]
        )
        print("\n5) 생성된 row 가 없습니다. (새로 처리된 파일이 없거나 유효 로그가 없음)")

    # 6) 이력 CSV 업데이트
    if new_history_rows:
        new_hist_df = pd.DataFrame(new_history_rows)
        if history_df is not None:
            updated_history = pd.concat([history_df, new_hist_df], ignore_index=True)
        else:
            updated_history = new_hist_df

        HISTORY_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
        updated_history.to_csv(HISTORY_CSV_PATH, index=False, encoding="utf-8-sig")
        print(f"6) 이력 CSV 업데이트 완료 → {HISTORY_CSV_PATH}")
    else:
        print("6) 추가로 기록할 이력이 없습니다. (새로 처리된 파일 없음)")

    # 7) 샘플(상위 1000행) 엑셀 저장 (index 컬럼 없이)
    if len(df) > 0:
        SAMPLE_EXCEL_PATH.parent.mkdir(parents=True, exist_ok=True)
        sample_df = df.head(1000)
        sample_df.to_excel(SAMPLE_EXCEL_PATH, index=False)
        print(f"7) 상위 1000행 샘플 엑셀 저장 완료 → {SAMPLE_EXCEL_PATH}")
    else:
        print("7) 저장할 데이터가 없어 샘플 엑셀은 생성하지 않았습니다.")

    # 8) 전체 df 반환
    return df


# =========================
# 실행
# =========================
df_full = main()
df_full.head(100)  # 노트북에서는 상위 100행만 표시


1) TXT 파일 스캔 완료 → 총 파일 수집: 4978개
2) 이력 CSV 없음 → 처음 실행으로 간주
3) 이번에 새로 처리할 대상 파일 수: 4978개
4) TXT 파일 처리 시작...
   → 현재 1000/4978 파일 처리 완료
   → 현재 2000/4978 파일 처리 완료
   → 현재 3000/4978 파일 처리 완료
   → 현재 4000/4978 파일 처리 완료
   → 현재 4978/4978 파일 처리 완료
4) TXT 파일 처리 완료

5) 총 생성된 row 수: 1172186개
6) 이력 CSV 업데이트 완료 → C:\Users\user\Desktop\c1_fct_testlog_detail_jason_processing.csv
7) 상위 1000행 샘플 엑셀 저장 완료 → C:\Users\user\Desktop\c1_fct_testlog_detail_result_sample1000.xlsx


,YYYYMMDD,End time,Barcode information,Test_item,Test_Time,Test_item_CT
0,20251001,08:24:01,BA1WJ24068500081SJ8T-14F014-AC,MES 이전공정 체크 SKIP,08:23:48.24,NaN
1,20251001,08:24:01,BA1WJ24068500081SJ8T-14F014-AC,START :: DMM SET CURRENT RANGE,08:23:50.61,2.37
2,20251001,08:24:01,BA1WJ24068500081SJ8T-14F014-AC,MODE SET :: CURR_RANG,08:23:50.63,0.02
3,20251001,08:24:01,BA1WJ24068500081SJ8T-14F014-AC,테스트 결과 : OK,08:23:50.86,0.23
4,20251001,08:24:01,BA1WJ24068500081SJ8T-14F014-AC,START :: DO SET,08:23:50.88,0.02
...,...,...,...,...,...,...
95,20251001,20:33:16,BA1WJ24068500081SJ8T-14F014-AC,Return Value :: OK,20:33:06.61,0.02
96,20251001,20:33:16,BA1WJ24068500081SJ8T-14F014-AC,테스트 결과 : OK,20:33:06.63,0.02
97,20251001,20:33:16,BA1WJ24068500081SJ8T-14F014-AC,START :: DO SET,20:33:06.65,0.02
98,20251001,20:33:16,BA1WJ24068500081SJ8T-14F014-AC,DO_SET Value :: 0000000000000000,20:33:06.67,0.02
